## 0. Colab Setup (run this first if on Google Colab)

Skip this section if running locally.

In [ ]:
# --- Run this cell ONLY on Google Colab ---
# Clone the repo and install dependencies

import subprocess, sys

# Clone repository
REPO_URL = 'https://github.com/manaranjanp/absa_adk.git'  # <-- Update with your repo URL
PROJECT_DIR = '/content/absa'

subprocess.run(['git', 'clone', REPO_URL, PROJECT_DIR], check=True)

# Change working directory to the project root
import os
os.chdir(PROJECT_DIR)

# Install dependencies
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

# Set API key (or use Colab Secrets)
# from google.colab import userdata
# os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY'] = 'your-api-key-here'  # <-- Replace this

print(f'Project cloned to {PROJECT_DIR}')
print('Setup complete!')

# ABSA Agent Pipeline — Interactive Notebook

This notebook demonstrates the **Aspect-Based Sentiment Analysis (ABSA)** multi-agent pipeline built with Google ADK.

The pipeline processes restaurant customer reviews through 6 sequential stages:
1. **Toxicity Detection** — Screens for toxic/abusive content
2. **Aspect Extraction** — Identifies restaurant experience aspects
3. **Sentiment Analysis** — Classifies sentiment per aspect
4. **Policy Evaluation & Escalation** — Checks alerts and creates tickets
5. **Response Generation** — Composes customer-facing response
6. **Output Guardrail** — Validates response before publishing

## 1. Setup & Configuration

In [ ]:
import os
import sys
import json

# Ensure project root is on the path
PROJECT_ROOT = os.path.dirname(os.path.abspath('__file__'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

# Configure observability (LangSmith tracing)
from observability import setup_observability, setup_litellm_logging
setup_observability()
setup_litellm_logging(verbose=False)

print('Setup complete.')

## 2. Escalation Tickets

Escalation tickets are stored as a JSON file at `data/escalation_tickets.json`. The file is created automatically on first use.

In [ ]:
import os

tickets_path = os.path.join('data', 'escalation_tickets.json')
if os.path.exists(tickets_path):
    print(f'Escalation tickets file exists at: {tickets_path}')
else:
    print(f'Escalation tickets file will be created at: {tickets_path} (on first escalation)')

## 3. Create the Pipeline

Build the SequentialAgent pipeline with your chosen LLM provider.

**Supported model strings:**
- `openai/gpt-4o` — OpenAI GPT-4o
- `anthropic/claude-3-5-sonnet-20241022` — Anthropic Claude
- `groq/llama3-8b-8192` — Groq (Llama 3)
- `xai/grok-2` — xAI Grok

In [ ]:
from pipeline.absa_pipeline import create_absa_pipeline
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner

# Choose your model
MODEL_NAME = os.getenv('DEFAULT_MODEL', 'openai/gpt-4o')
print(f'Using model: {MODEL_NAME}')

# Create pipeline and runner
pipeline = create_absa_pipeline(MODEL_NAME)
session_service = InMemorySessionService()
runner = Runner(
    agent=pipeline,
    app_name='absa_pipeline_app',
    session_service=session_service,
)

print(f'Pipeline created with {len(pipeline.sub_agents)} sub-agents:')
for i, agent in enumerate(pipeline.sub_agents):
    print(f'  [{i+1}] {agent.name}')

## 4. Load Sample Reviews

In [ ]:
with open('data/sample_reviews.json', 'r') as f:
    all_reviews = json.load(f)

print(f'Loaded {len(all_reviews)} sample reviews')
print(f'\nFirst review preview:')
print(json.dumps(all_reviews[0], indent=2))

## 5. Run Pipeline — Single Review

Process one review through the full 6-stage pipeline.

In [ ]:
from pipeline.absa_pipeline import run_pipeline

# Pick a review (change index to try different ones)
# REV-001: Positive multi-aspect review
review = all_reviews[0]
print(f'Processing: {review["review_id"]}')
print(f'Text: {review["review_text"][:100]}...')

result = await run_pipeline(
    review,
    model_name=MODEL_NAME,
    pipeline=pipeline,
    session_service=session_service,
    runner=runner,
)

### Inspect Stage Results

In [ ]:
print('=== Toxicity Result ===')
print(json.dumps(result.get('toxicity_result', {}), indent=2))

print('\n=== Aspect Extraction ===')
print(json.dumps(result.get('aspect_result', {}), indent=2))

print('\n=== Sentiment Analysis ===')
print(json.dumps(result.get('sentiment_result', {}), indent=2))

In [ ]:
print('=== Escalation Result ===')
print(json.dumps(result.get('escalation_result', {}), indent=2))

print('\n=== Generated Response ===')
resp = result.get('response_result', {})
print(resp.get('response_text', 'No response generated'))

print('\n=== Guardrail Result ===')
print(json.dumps(result.get('guardrail_result', {}), indent=2))

## 6. Run Pipeline — Toxic Review

Demonstrate that toxic reviews are terminated at Stage 1 with no response.

In [ ]:
# REV-004 and REV-024 are toxic reviews in the dataset
toxic_review = all_reviews[3]  # REV-004
print(f'Processing toxic review: {toxic_review["review_id"]}')
print(f'Text: {toxic_review["review_text"][:100]}...')

toxic_result = await run_pipeline(
    toxic_review,
    model_name=MODEL_NAME,
    pipeline=pipeline,
    session_service=session_service,
    runner=runner,
)

## 7. Run Pipeline — Negative Review with Escalation

Process a negative review from a Platinum customer to trigger escalation.

In [ ]:
# REV-009: Negative review from Platinum customer
escalation_review = all_reviews[8]  # REV-009
print(f'Processing: {escalation_review["review_id"]}')
print(f'Tier: {escalation_review["customer_tier"]}')
print(f'Text: {escalation_review["review_text"][:100]}...')

esc_result = await run_pipeline(
    escalation_review,
    model_name=MODEL_NAME,
    pipeline=pipeline,
    session_service=session_service,
    runner=runner,
)

## 8. Run Batch — Multiple Reviews

Process a small batch of reviews to see the full pipeline in action.

In [ ]:
import random
import time

# Select 5 random reviews
batch = random.sample(all_reviews, 5)
print(f'Selected {len(batch)} reviews for batch processing:\n')
for r in batch:
    print(f'  {r["review_id"]} | {r["customer_tier"]} | {r["review_text"][:60]}...')

batch_results = []
start = time.time()

for review in batch:
    result = await run_pipeline(
        review,
        model_name=MODEL_NAME,
        pipeline=pipeline,
        session_service=session_service,
        runner=runner,
    )
    batch_results.append(result)

elapsed = time.time() - start
print(f'\nBatch complete: {len(batch)} reviews in {elapsed:.1f}s ({elapsed/len(batch):.1f}s avg)')

## 9. Inspect Escalation Tickets

Check the escalation tickets stored in `data/escalation_tickets.json`.

In [ ]:
import json

tickets_path = 'data/escalation_tickets.json'
if os.path.exists(tickets_path):
    with open(tickets_path, 'r') as f:
        lines = f.readlines()
    tickets = [json.loads(line) for line in lines if line.strip()]
    print(f'=== Escalation Tickets ({len(tickets)} total) ===')
    for ticket in tickets:
        print(f'  {json.dumps(ticket, indent=2)}')
else:
    print('No escalation tickets file found yet.')

## 10. Inspect Human Review Queue

Check responses that were flagged by the guardrail and queued for human review.

In [ ]:
import json
import os

human_review_path = 'data/human_review_queue.json'
if os.path.exists(human_review_path):
    with open(human_review_path, 'r') as f:
        lines = f.readlines()
    entries = [json.loads(line) for line in lines if line.strip()]
    print(f'=== Human Review Queue ({len(entries)} entries) ===')
    for entry in entries:
        print(json.dumps(entry, indent=2))
        print()
else:
    print('No responses queued for human review.')